# Fine-tuning CamemBERT — détecteur de fin de tour (EOU) français

Objectif : classer un **partiel ASR français** en `fini` (le locuteur a terminé son tour) vs `pas_fini` (il parle encore).
Sortie = **`P(fini)`** → c'est le **`p_lex`** de la branche sémantique (pipe 2), branché derrière l'interface `LexicalEOU` du POC (drop-in de `FrenchSemanticEOU`).

## TP / TN / FP / FN dans NOTRE contexte
Classe positive = **`fini`** (= déclencher la réponse de l'agent / endpoint).

| | Vérité = `fini` | Vérité = `pas_fini` |
|---|---|---|
| **Prédit `fini`** | ✅ **TP** — on répond au bon moment | 🛑 **FP — ON COUPE LE PATIENT** (épellation, n°, hésitation) ← *erreur à minimiser* |
| **Prédit `pas_fini`** | ⏳ **FN** — on attend trop / blanc / lag | ✅ **TN** — on écoute, zéro interruption |

Au téléphone médical : **FP (couper la parole) ≫ FN (latence)** en gravité.
→ on optimise la **précision sur `fini`** (peu de FP), quitte à tolérer des FN.
Le seuil final vit dans la fusion (`veto_lex` / `mid`), pas dans le modèle.

In [ ]:
# Colab / environnement neuf — décommenter :
# !pip install -q transformers datasets torch scikit-learn accelerate "optimum[onnxruntime]"

In [ ]:
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)

set_seed(42)
MODEL_NAME = "almanach/camembert-base"   # FR-natif, licence MIT
ID2LABEL = {0: "pas_fini", 1: "fini"}    # classe positive = 1 (fini)
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
MAX_LEN = 64                             # énoncés courts

## 1. Données

⚠️ Ci-dessous un **dataset jouet** (~60 ex.) juste pour valider le pipeline de bout en bout.
À remplacer par tes vraies données (transcripts d'appels + labelling Azure LLM).
Il couvre déjà les cas durs FR : épellation, numéros/dates en cours, hésitations, connecteurs.

**Règle d'or au passage en vrai dataset : split par APPEL, pas par ligne** (sinon fuite de données = métriques mensongères).

In [ ]:
FINI = [  # label 1 — tour terminé
    "oui c'est exact", "je voudrais prendre un rendez-vous pour demain matin",
    "merci beaucoup au revoir", "d'accord c'est noté", "non je n'ai pas d'allergie",
    "mon médecin traitant est le docteur martin", "c'est pour un renouvellement d'ordonnance",
    "oui tout à fait", "je confirme le rendez-vous de jeudi", "très bien merci",
    "c'est ça", "j'ai mal à la tête depuis ce matin", "le rendez-vous est annulé",
    "parfait à bientôt", "je voudrais parler à un médecin", "non c'est tout",
    "oui je suis disponible vendredi", "bonjour je vous appelle pour un résultat d'analyse",
    "d'accord je rappellerai plus tard", "c'est noté merci beaucoup",
    "je prends le créneau de quatorze heures", "oui c'est bien moi",
    "non merci ça ira", "le patient s'appelle dupont", "je voudrais annuler ma consultation",
    "oui j'ai bien compris", "c'est urgent", "à demain docteur", "voilà c'est tout pour moi",
    "je vous remercie",
]

PAS_FINI = [  # label 0 — encore en train de parler
    "je voudrais prendre un", "alors euh", "mon nom c'est d u", "le rendez-vous c'est le",
    "mon numéro est zéro six", "est-ce que je peux", "donc je", "c'est pour",
    "je m'appelle martin d comme", "le code postal c'est soixante quinze",
    "ma date de naissance c'est le douze", "je voudrais savoir si", "attendez je cherche mon",
    "c'est à propos de", "il faut que je", "mon médecin c'est le docteur",
    "je dois vous donner mon numéro de", "alors c'est le quinze", "euh comment dire",
    "je vais vous épeler mon nom c'est m a r", "le résultat de mon analyse de",
    "j'ai rendez-vous avec le", "pouvez-vous me dire si", "ça fait depuis",
    "et ensuite", "mon adresse c'est douze rue", "je prends le créneau de",
    "c'est pour mon fils qui a", "la posologie c'est", "je vous appelle parce que",
]

texts  = FINI + PAS_FINI
labels = [1] * len(FINI) + [0] * len(PAS_FINI)

tr_t, te_t, tr_y, te_y = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)
dsd = DatasetDict({
    "train": Dataset.from_dict({"text": tr_t, "label": tr_y}),
    "test":  Dataset.from_dict({"text": te_t, "label": te_y}),
})
print(dsd)

## 2. Tokenizer + modèle

CamemBERT comprend déjà le français (pré-entraîné). On ajoute une **tête de classification** neuve (2 sorties) qu'on entraîne.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok_fn(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LEN)

dst = dsd.map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)

## 3. Métriques — dans NOTRE contexte

On expose `TP / TN / FP / FN` avec leur sens métier : **FP = interruptions** (à minimiser), **FN = latence**.
On choisit `precision_fini` comme métrique de sélection du meilleur modèle (= le moins d'interruptions).

In [ ]:
def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = np.argmax(logits, axis=-1)
    acc = accuracy_score(y, pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": acc, "precision_fini": p, "recall_fini": r, "f1": f1,
            "TP": int(tp), "TN": int(tn),
            "FP_interruptions": int(fp), "FN_latence": int(fn)}

## 4. Entraînement

~5 epochs suffisent sur ce jouet. Sur ton vrai dataset (2-10k ex.) : quelques minutes sur GPU (Colab T4), ~30-60 min CPU.

In [ ]:
args = TrainingArguments(
    output_dir="camembert-eou",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",          # transformers <4.40 : evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="precision_fini",   # priorité : peu d'interruptions
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=dst["train"], eval_dataset=dst["test"],
    processing_class=tok, compute_metrics=compute_metrics,
)
trainer.train()

## 5. Évaluation + analyse d'erreurs → améliorer le dataset

⚠️ Sur ce jouet (~15 ex. de test) les chiffres sont **bruités** — c'est juste pour valider le flux.
La vraie valeur ici : **lister les FP et FN concrets**. Chaque erreur = un type d'exemple à ajouter (bien labellisé) au dataset. C'est la boucle d'amélioration (active learning).

In [ ]:
metrics = trainer.evaluate()
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in metrics.items()})

pred = np.argmax(trainer.predict(dst["test"]).predictions, axis=-1)
test = dst["test"]

print("\n🛑 FP — prédit 'fini' mais PAS fini (= ON COUPE LE PATIENT) :")
for t, yt, yp in zip(test["text"], test["label"], pred):
    if yp == 1 and yt == 0:
        print("   ", t)

print("\n⏳ FN — prédit 'pas_fini' mais fini (= LATENCE / blanc) :")
for t, yt, yp in zip(test["text"], test["label"], pred):
    if yp == 0 and yt == 1:
        print("   ", t)

## 6. Inférence — `P(fini)` = `p_lex`

C'est la fonction qui alimentera la fusion du POC.

In [ ]:
import torch

def p_fini(text: str) -> float:
    enc = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1)[0, 1].item()   # P(fini) = p_lex

for t in ["je voudrais prendre un", "oui c'est exact",
          "mon numéro c'est zéro six", "merci au revoir",
          "mon nom c'est m a r"]:
    print(f"  p_fini={p_fini(t):.2f}  |  {t}")

## 7. Export ONNX + intégration POC

Quand les métriques te conviennent :

```bash
optimum-cli export onnx --model camembert-eou camembert-eou-onnx/
# (option) quantize int8 → ~10ms CPU
```

Puis dans le POC, derrière l'interface `LexicalEOU` (drop-in de `FrenchSemanticEOU`) :

```python
# eou_detector/eou/camembert.py
class CamembertLexicalEOU(LexicalEOU):
    def __init__(self, onnx_dir): ...   # onnxruntime + tokenizer
    def predict(self, text):
        p = self._p_fini(text)          # softmax -> P(fini)
        return LexResult(p_lex=p, veto=False, reason="camembert")
```

**Ensemble recommandé** : garder les **règles** (`FrenchSemanticEOU`) comme **veto dur** (épellation / numéros) + le **modèle** pour la complétude ouverte. Mieux que l'un seul.

## Étapes suivantes (prochaine session)
1. **Vrai dataset** : transcripts d'appels (Mongo/Psql) → tours complets = `fini`, tronqués = `pas_fini` ; + labelling Azure LLM. **Split par appel.**
2. ~2-10k ex. équilibrés, couvrant épellation / numéros / dates / hésitations / confirmations.
3. Itérer via l'analyse FP/FN (cellule 5).
4. **Battre le baseline règles** sur la complétude ouverte ; comparer FP/FN des deux.